# ricetein smoke test (Kaggle plumbing)

Checks, in order: the three datasets are mounted, the locked package versions install,
our code imports, the GPU works, and a run folder is written with `RunRecorder`.
Every Kaggle notebook in the project starts with cells 1–3 of this notebook.

Run it with `./scripts/kaggle_push.sh 00-smoke`, then `./scripts/kaggle_pull.sh 00-smoke`.

In [ ]:
# 1. Find the mounted inputs. Kaggle mounts datasets at /kaggle/input/datasets/<owner>/<slug>
#    (older layout: /kaggle/input/<slug>), so search both. Do NOT import torch before cell 2.
from pathlib import Path

def find_input(slug):
    root = Path("/kaggle/input")
    if (root / slug).is_dir():
        return root / slug
    hits = sorted(p for p in root.glob(f"*/*/{slug}") if p.is_dir())
    return hits[0] if hits else None

CODE_DIR = find_input("ricetein-code")
PRIMARY_DIR = find_input("rice-diseases-image-dataset")
PADDY_DIR = find_input("paddy-doctor-full")
for name, path in [("code", CODE_DIR), ("primary", PRIMARY_DIR), ("paddy", PADDY_DIR)]:
    print(f"{name:8s} -> {path}")
    assert path is not None, f"{name} dataset is not attached (check dataset_sources in kernel-metadata.json)"
print("code version:", (CODE_DIR / "CODE_VERSION").read_text().strip())

In [ ]:
# 2. Install the exact locked versions (same as the laptop's uv.lock), then make our code importable.
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements-kaggle.txt")],
    check=True,
)
sys.path.insert(0, str(CODE_DIR))

In [ ]:
# 3. Check the installed versions match the lock file and the GPU is usable.
import re
from importlib import metadata

locked = dict(re.findall(r"^([A-Za-z0-9_.-]+)==(\S+)", (CODE_DIR / "requirements-kaggle.txt").read_text(), re.M))
mismatch = {}
for pkg in ["torch", "torchvision", "timm", "transformers", "numpy", "scikit-learn"]:
    have = metadata.version(pkg)
    print(f"{pkg:14s} installed {have:20s} locked {locked.get(pkg)}")
    if have != locked.get(pkg):
        mismatch[pkg] = (have, locked.get(pkg))
assert not mismatch, f"Installed versions differ from the lock: {mismatch}"

import torch
assert torch.cuda.is_available(), "No GPU: check enable_gpu / accelerator"
print("GPU:", torch.cuda.get_device_name(0), "| CUDA", torch.version.cuda)
x = torch.randn(1024, 1024, device="cuda")
print("GPU matmul ok:", float((x @ x).sum()) == float((x @ x).sum()))

In [ ]:
# 4. Peek at both datasets.
labelled = PRIMARY_DIR / "LabelledRice" / "Labelled"
for cls in sorted(p.name for p in labelled.iterdir()):
    print(f"primary {cls:10s} {sum(1 for _ in (labelled / cls).glob('*.jpg')):5d} images")
paddy_root = next((p for p in [PADDY_DIR, *PADDY_DIR.glob('*')] if (p / "metadata.csv").exists()), None)
assert paddy_root is not None, "metadata.csv not found in the Paddy Doctor dataset"
print("paddy root:", paddy_root, "| classes:", sum(1 for p in paddy_root.iterdir() if p.is_dir()))

In [ ]:
# 5. Write a run folder exactly like a real experiment would.
from src.records import RunRecorder

rec = RunRecorder("smoke_test", config={"purpose": "kaggle plumbing check"}, overwrite=True)
rec.log_epoch(1, {"dummy_metric": 1.0})
rec.save_metrics({"smoke": "passed", "partition": "none"})
record = rec.finish()
print("code_version in record:", record["code_version"])
assert record["code_version"] != "unknown", "CODE_VERSION was not found"
print("Smoke test passed.")